# Arabic Summarizer Fine-tuning
**Model:** `UBC-NLP/AraT5-base`  
**Dataset:** `wiki_lingua` Arabic split (confirmed working)  
**Runtime:** Google Colab T4 GPU

In [1]:
# Cell 1: Install
!pip install -q transformers datasets accelerate rouge-score sentencepiece
print('Done')

  Preparing metadata (setup.py) ... done
Done


In [2]:
# Cell 2: Imports & GPU check
import os, gc, warnings
import numpy as np
import torch
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, EarlyStoppingCallback,
)
from rouge_score import rouge_scorer as _rouge_scorer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    raise SystemError('GPU not detected! Go to Runtime → Change runtime type → T4 GPU')

Device: cuda


In [3]:
# Cell 3: Load XL-Sum from Kaggle
import kagglehub
from kagglehub import KaggleDatasetAdapter

df_all = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yashdogra/xlsum-punjabi",
    "xlsum_all_train.csv",
)
print("All languages:", df_all["language"].unique())
print("Shape:", df_all.shape)

100%|██████████| 5.35G/5.35G [01:42<00:00, 55.9MB/s]


All languages: ['amharic' 'arabic' 'azerbaijani' 'bengali' 'burmese' 'chinese_simplified'
 'chinese_traditional' 'english' 'french' 'gujarati' 'hausa' 'hindi'
 'igbo' 'indonesian' 'japanese' 'kirundi' 'korean' 'kyrgyz' 'marathi'
 'nepali' 'oromo' 'pashto' 'persian' 'pidgin' 'portuguese' 'punjabi'
 'russian' 'scottish_gaelic' 'serbian_cyrillic' 'serbian_latin' 'sinhala'
 'somali' 'spanish' 'swahili' 'tamil' 'telugu' 'thai' 'tigrinya' 'turkish'
 'ukrainian' 'urdu' 'uzbek' 'vietnamese' 'welsh' 'yoruba']
Shape: (1122857, 6)


In [4]:
# Cell 4: Filter Arabic and split
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset as HFDataset

# Filter Arabic only
df = df_all[df_all["language"] == "arabic"][["text", "summary"]].dropna()
df = df.rename(columns={"text": "article"})
df = df[(df["article"].str.len() > 50) & (df["summary"].str.len() > 10)]
print(f"Arabic rows: {len(df)}")
print("\nSample article:", df.iloc[0]["article"][:300])
print("Sample summary:", df.iloc[0]["summary"])

# Sample and split
df = df.sample(n=min(5000, len(df)), random_state=42).reset_index(drop=True)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)

train_ds = HFDataset.from_pandas(train_df.reset_index(drop=True))
val_ds   = HFDataset.from_pandas(val_df.reset_index(drop=True))
test_ds  = HFDataset.from_pandas(test_df.reset_index(drop=True))

print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Arabic rows: 37509

Sample article: وكان الرئيس الأوكراني المؤقت، الكسندر تورتشينوف، قد أمر بسحب جميع القوات الأوكرانية من القرم. وسيطرت قوات روسية صباح الاثنين على قاعدة بحرية أوكرانية في فيودوسيا، في ثالث هجوم من نوعه خلال 48 ساعة، وذلك بحسب تصريحات مسؤولين أوكرانيين لبي بي سي . وقال المتحدث باسم وزارة الدفاع الأوكرانية فلاديسلاف سي
Sample summary: بدأت القوات الأوكرانية الانسحاب من شبه جزيرة القرم.

Train: 4000 | Val: 500 | Test: 500


In [5]:
# Cell 6: Tokenizer & preprocessing
from transformers import AutoTokenizer

SUM_MODEL_NAME = 'google/mt5-small'
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 128

print(f'Loading tokenizer: {SUM_MODEL_NAME}')
sum_tokenizer = AutoTokenizer.from_pretrained(SUM_MODEL_NAME, use_fast=False)

def preprocess(batch):
    model_inputs = sum_tokenizer(
        batch['article'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = sum_tokenizer(
        text_target=batch['summary'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

REMOVE_COLS = ['article', 'summary', '__index_level_0__']
def safe_remove(ds, cols):
    return [c for c in cols if c in ds.column_names]

print('Tokenizing...')
tok_train = train_ds.map(preprocess, batched=True, remove_columns=safe_remove(train_ds, REMOVE_COLS))
tok_val   = val_ds.map(preprocess,   batched=True, remove_columns=safe_remove(val_ds,   REMOVE_COLS))
tok_test  = test_ds.map(preprocess,  batched=True, remove_columns=safe_remove(test_ds,  REMOVE_COLS))
print('Done')

Loading tokenizer: google/mt5-small


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Tokenizing...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Done


In [6]:
# Cell 7: Free memory
gc.collect()
torch.cuda.empty_cache()
print('Memory cleared')

Memory cleared


In [7]:
# Cell 7: Simple Arabic ROUGE (no dependency issues)
import numpy as np
import re
from collections import Counter
from rouge_score import scoring

def arabic_tokenize(text):
    # Remove punctuation, normalize, split on whitespace
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def _ngrams(tokens, n):
    c = Counter()
    for ng in (tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)):
        c[ng] += 1
    return c

def _lcs_len(a, b):
    t = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            t[i][j] = t[i-1][j-1]+1 if a[i-1]==b[j-1] else max(t[i-1][j], t[i][j-1])
    return t[-1][-1]

def rouge_n(ref, hyp, n):
    r = _ngrams(ref, n)
    h = _ngrams(hyp, n)
    inter = sum(min(r[ng], h[ng]) for ng in r)
    prec = inter / max(sum(h.values()), 1)
    rec  = inter / max(sum(r.values()), 1)
    f = 2*prec*rec / max(prec+rec, 1e-9)
    return f

def rouge_l(ref, hyp):
    l = _lcs_len(ref, hyp)
    prec = l / max(len(hyp), 1)
    rec  = l / max(len(ref), 1)
    f = 2*prec*rec / max(prec+rec, 1e-9)
    return f

def compute_rouge(eval_pred):
    predictions, labels = eval_pred
    if predictions.ndim == 3:
        predictions = predictions.argmax(axis=-1)
    predictions = np.clip(predictions, 0, sum_tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, sum_tokenizer.pad_token_id)
    decoded_preds  = sum_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = sum_tokenizer.batch_decode(labels,      skip_special_tokens=True)
    r1, r2, rL = [], [], []
    for p, l in zip(decoded_preds, decoded_labels):
        ref = arabic_tokenize(l)
        hyp = arabic_tokenize(p)
        r1.append(rouge_n(ref, hyp, 1))
        r2.append(rouge_n(ref, hyp, 2))
        rL.append(rouge_l(ref, hyp))
    return {
        'rouge1': round(np.mean(r1) * 100, 4),
        'rouge2': round(np.mean(r2) * 100, 4),
        'rougeL': round(np.mean(rL) * 100, 4),
    }

# Sanity check
ref = arabic_tokenize('بدأت القوات الأوكرانية الانسحاب من شبه جزيرة القرم')
hyp = arabic_tokenize('انسحبت القوات الأوكرانية من القرم')
print('Sanity rouge1:', round(rouge_n(ref, hyp, 1)*100, 2))
print('Arabic ROUGE ready')

Sanity rouge1: 61.54
Arabic ROUGE ready


In [8]:
# Cell 8: Load model & trainer
import os
from transformers import (
    AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback,
)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Loading model: {SUM_MODEL_NAME}')
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_MODEL_NAME)
sum_model.gradient_checkpointing_enable()
sum_model.config.use_cache = False
sum_model.generation_config.forced_eos_token_id = sum_tokenizer.eos_token_id

sum_collator = DataCollatorForSeq2Seq(
    tokenizer=sum_tokenizer, model=sum_model,
    label_pad_token_id=-100, pad_to_multiple_of=8,
)

USE_BF16 = torch.cuda.is_bf16_supported()
fast_val  = tok_val.select(range(min(200, len(tok_val))))

training_args = Seq2SeqTrainingArguments(
    output_dir='./arat5-arabic-summarization',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim='adafactor',
    learning_rate=5e-4,
    warmup_ratio=0.05,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rouge1',
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=4,
    logging_steps=50,
    report_to=['none'],
    dataloader_num_workers=2,
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=sum_model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=fast_val,
    processing_class=sum_tokenizer,
    data_collator=sum_collator,
    compute_metrics=compute_rouge,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Trainer ready')

Loading model: google/mt5-small


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer ready


In [9]:
# Cell 10: TRAIN
print('Starting fine-tuning...')
trainer.train()
trainer.save_model('./arat5-arabic-summarization')
sum_tokenizer.save_pretrained('./arat5-arabic-summarization')
print('Training complete! Model saved.')

Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,17.421030,3.127586,15.224500,4.239400,13.238100
2,13.430627,2.851886,16.505700,4.692600,14.349700
3,12.640889,2.773808,17.479300,5.284400,15.080200
4,11.933065,2.721063,17.254100,5.493100,14.773000
5,11.315568,2.706349,18.375000,5.962700,15.830300
6,10.869579,2.694100,18.614800,6.019800,15.850300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,17.421030,3.127586,15.224500,4.239400,13.238100
2,13.430627,2.851886,16.505700,4.692600,14.349700
3,12.640889,2.773808,17.479300,5.284400,15.080200
4,11.933065,2.721063,17.254100,5.493100,14.773000
5,11.315568,2.706349,18.375000,5.962700,15.830300
6,10.869579,2.694100,18.614800,6.019800,15.850300


KeyboardInterrupt: 